In [2]:
##############################################################################
# ===== Setup: imports, hub connection, settings =====
##############################################################################
#
import json, warnings
warnings.filterwarnings("ignore", module="argopy")
warnings.filterwarnings("ignore", message="Engine 'argo' loading failed")
 
import numpy as np
import pandas as pd
import s3fs
import xarray as xr
 
PROFILE  = "hack-team"
fs = s3fs.S3FileSystem(profile=PROFILE)        
BUCKET   = "ocean-hackathon-2026"
DATA_DIR = f"{BUCKET}/data/elnino"             
TEAM_DIR = f"{BUCKET}/teams/team-elnino"       
# ------------------------------------------------------------------------------------
OUT_DIR  = f"{TEAM_DIR}/processed"
opts = {"profile": PROFILE}                    # for pandas/xarray: storage_options=opts
 
# Climatology baseline. Ending at 2018 keeps it identical to the model's training period,
# so the backtest on 2019+ never sees test years through the anomalies.
CLIM_YEARS = ("1993", "2018")
 
# Crop to the BC / US west coast. Keeps memory use under ~1 GB
# and keeps the processed data small enough to ship with the app.
# Set REGION = None for the full NEP box (25-80N, 170-100W) - needs a 16 GB server.
REGION = dict(latitude=slice(40, 62), longitude=slice(-150, -120))
 
VARS = {  # name in file -> (label, units, colormap hint for the frontend)
    "to":     ("Sea surface temperature", "°C",  "thermal"),
    "so":     ("Sea surface salinity",    "PSU", "haline"),
    "zo":     ("Sea surface height",      "m",   "balance"),
    "mlotst": ("Mixed layer depth",       "m",   "deep"),
    "ugo":    ("Geostrophic current (east)",  "m/s", "balance"),
    "vgo":    ("Geostrophic current (north)", "m/s", "balance"),
}

In [3]:
##############################################################################
# ===== Open and stitch the two NEP files =====
##############################################################################
# %%
# Open and stitch the reprocessed (1993-2024) and near-real-time (2025-present) files
def open_nep(name):
    ds = xr.open_dataset(fs.open(f"{DATA_DIR}/{name}"), engine="h5netcdf",
                         chunks={"time": 12})     # read a year at a time, not the whole file
    if "depth" in ds.dims:
        ds = ds.isel(depth=0, drop=True)          # surface only; mlotst has no depth dim
    return ds[list(VARS)]
 
old = open_nep("mob_armor3d_nep_surface_1993-2024.nc")
new = open_nep("mob_armor3d_nep_surface_2025-present.nc")
 
ds = xr.concat([old, new], dim="time")
ds = ds.sel(time=~ds.get_index("time").duplicated(keep="last"))   # prefer NRT if months overlap
ds = ds.sortby("time")
ds["time"] = ds.indexes["time"].to_period("M").to_timestamp()     # normalise to month start
 
if REGION:
    ds = ds.sel(**REGION)
ds = ds.astype("float32")
for v in ds.variables:
    ds[v].encoding = {}                                            # drop NetCDF encodings before Zarr
 
print(ds)
print("Months:", ds.time.values[0], "->", ds.time.values[-1], f"({ds.sizes['time']})")
print(f"In-memory size: {ds.nbytes / 1e6:.0f} MB")
 
# Months should be continuous; gaps break the lagged features later
expected = pd.date_range(ds.time.values[0], ds.time.values[-1], freq="MS")
missing = expected.difference(ds.indexes["time"])
print("Missing months:", list(missing.strftime("%Y-%m")) or "none")

<xarray.Dataset> Size: 410MB
Dimensions:    (time: 404, latitude: 176, longitude: 240)
Coordinates:
  * latitude   (latitude) float32 704B 40.06 40.19 40.31 ... 61.69 61.81 61.94
  * longitude  (longitude) float32 960B -149.9 -149.8 -149.7 ... -120.2 -120.1
  * time       (time) datetime64[us] 3kB 1993-01-01 1993-02-01 ... 2026-08-01
Data variables:
    to         (time, latitude, longitude) float32 68MB dask.array<chunksize=(12, 176, 240), meta=np.ndarray>
    so         (time, latitude, longitude) float32 68MB dask.array<chunksize=(12, 176, 240), meta=np.ndarray>
    zo         (time, latitude, longitude) float32 68MB dask.array<chunksize=(12, 176, 240), meta=np.ndarray>
    mlotst     (time, latitude, longitude) float32 68MB dask.array<chunksize=(12, 176, 240), meta=np.ndarray>
    ugo        (time, latitude, longitude) float32 68MB dask.array<chunksize=(12, 176, 240), meta=np.ndarray>
    vgo        (time, latitude, longitude) float32 68MB dask.array<chunksize=(12, 176, 240), meta=

In [4]:
##############################################################################
# ===== Load into memory + climatology =====
##############################################################################
# %%
# Load into memory one variable at a time, single-threaded, so peak memory stays low
import dask, gc
with dask.config.set(scheduler="synchronous"):
    loaded = {}
    for v in VARS:
        loaded[v] = ds[v].load()
        print(f"loaded {v}: {loaded[v].nbytes / 1e6:.0f} MB")
        gc.collect()
ds = xr.Dataset(loaded)
print(f"Total in memory: {ds.nbytes / 1e6:.0f} MB")
clim = ds.sel(time=slice(*CLIM_YEARS)).groupby("time.month").mean("time")

loaded to: 68 MB
loaded so: 68 MB
loaded zo: 68 MB
loaded mlotst: 68 MB
loaded ugo: 68 MB
loaded vgo: 68 MB
Total in memory: 410 MB


In [5]:
##############################################################################
# ===== Seam check at the 2024 -> 2025 boundary =====
##############################################################################
# %%
# Seam check: does the 2024 -> 2025 file boundary jump?
# Domain-mean anomaly for the 6 months either side. A step change in Jan 2025 that isn't
# in the neighbouring months means the NRT product is biased against the reprocessed one.
anom = ds.groupby("time.month") - clim
seam = anom.sel(time=slice("2024-07", "2025-06")).mean(["latitude", "longitude"])
print(seam.to_dataframe().drop(columns="month", errors="ignore").round(3))

               to     so     zo  mlotst    ugo    vgo
time                                                 
2024-07-01  0.237 -0.083  0.011  -0.569  0.002 -0.003
2024-08-01  0.658 -0.063  0.017  -0.533  0.001 -0.002
2024-09-01  0.584 -0.030  0.011  -0.007  0.002 -0.005
2024-10-01  0.164 -0.012  0.014   3.440  0.000 -0.002
2024-11-01 -0.227 -0.026  0.009   6.312  0.002 -0.003
2024-12-01  0.253 -0.084  0.021   2.655 -0.002  0.000
2025-01-01  0.529 -0.088  0.042   1.912  0.002 -0.004
2025-02-01  0.514 -0.097  0.033   3.788  0.002 -0.004
2025-03-01  0.668 -0.145  0.023   1.423 -0.000 -0.001
2025-04-01  0.692 -0.106  0.009  -3.966 -0.001 -0.002
2025-05-01  0.560 -0.140  0.016   1.211  0.000 -0.002
2025-06-01  0.431 -0.131  0.014   0.671  0.002 -0.004


In [6]:
##############################################################################
# ===== Write Zarr to the team bucket =====
##############################################################################
# %%
# Write Zarr to the team bucket
chunks = {"time": 12, "latitude": -1, "longitude": -1}   # 1 year per chunk; fine for maps and point series
ds.chunk(chunks).to_zarr(f"s3://{OUT_DIR}/ocean_monthly.zarr", mode="w",
                          storage_options=opts, consolidated=True)
clim.chunk({"month": 12}).to_zarr(f"s3://{OUT_DIR}/climatology.zarr", mode="w",
                                   storage_options=opts, consolidated=True)
 
def du(path):
    return sum(f["size"] for f in fs.find(path, detail=True).values()) / 1e6
print(f"ocean_monthly.zarr on S3: {du(f'{OUT_DIR}/ocean_monthly.zarr'):.0f} MB (compressed)")
print(f"climatology.zarr on S3:   {du(f'{OUT_DIR}/climatology.zarr'):.0f} MB")

/home/.pixi/envs/default/lib/python3.14/site-packages/zarr/api/asynchronous.py:246: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/home/.pixi/envs/default/lib/python3.14/site-packages/zarr/api/asynchronous.py:246: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


ocean_monthly.zarr on S3: 105 MB (compressed)
climatology.zarr on S3:   6 MB


In [10]:
##############################################################################
# ===== ENSO index from NOAA ONI =====
##############################################################################
# %%
# ENSO: official NOAA ONI (3-month running mean of Niño 3.4 SST anomaly)
# Season label is centred on its middle month: DJF -> Jan, JFM -> Feb, ..., NDJ -> Dec.
SEASON_MID = {"DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4, "AMJ": 5, "MJJ": 6,
              "JJA": 7, "JAS": 8, "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12}
oni = pd.read_csv("https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt", sep=r"\s+")
oni["time"] = pd.to_datetime(dict(year=oni["YR"], month=oni["SEAS"].map(SEASON_MID), day=1))
oni = oni.set_index("time")["ANOM"].rename("oni").sort_index()
 
def label_events(s, thresh=0.5, run=5):
    """NOAA definition: El Niño/La Niña when ONI >= +0.5 / <= -0.5 for >= 5 consecutive seasons.
    A run still going at the end of the record can't have reached 5 seasons yet, so it is
    labelled with its phase and flagged provisional (NOAA would call these 'El Niño conditions')."""
    phase = pd.Series("neutral", index=s.index)
    provisional = pd.Series(False, index=s.index)
    for sign, name in ((1, "el_nino"), (-1, "la_nina")):
        hit = (sign * s) >= thresh
        grp = (hit != hit.shift()).cumsum()
        size = hit.groupby(grp).transform("size")
        confirmed = hit & size.ge(run)
        ongoing = hit & (grp == grp.iloc[-1]) & size.lt(run)
        phase[confirmed | ongoing] = name
        provisional[ongoing] = True
    return phase, provisional
 
phase, provisional = label_events(oni)
enso = pd.DataFrame({"oni": oni, "phase": phase, "provisional": provisional}).loc["1993":]
enso_records = [{"month": t.strftime("%Y-%m"), "oni": round(float(r.oni), 2),
                 "phase": r.phase, "provisional": bool(r.provisional)}
                for t, r in enso.iterrows()]
with fs.open(f"{OUT_DIR}/enso.json", "w") as f:
    json.dump(enso_records, f)
print(enso.tail(6))

             oni    phase  provisional
time                                  
2026-03-01  0.11  neutral        False
2026-04-01  0.46  neutral        False
2026-05-01  0.95  el_nino         True
2026-06-01  1.39  el_nino         True
2026-07-01  1.80  el_nino         True
2026-08-01  2.16  el_nino         True


In [11]:
##############################################################################
# ===== meta.json for the frontend =====
##############################################################################
# %%
# meta.json - the frontend reads this instead of hard-coding months/variables/ranges
def robust_range(da):
    lo, hi = np.nanpercentile(da.values, [2, 98])
    return [round(float(lo), 3), round(float(hi), 3)]
 
meta = {
    "first_month": pd.Timestamp(ds.time.values[0]).strftime("%Y-%m"),
    "last_observed_month": pd.Timestamp(ds.time.values[-1]).strftime("%Y-%m"),
    "climatology_years": list(CLIM_YEARS),
    "bounds": {  # [west, south, east, north] for MapLibre image overlays
        "west": float(ds.longitude.min()), "east": float(ds.longitude.max()),
        "south": float(ds.latitude.min()), "north": float(ds.latitude.max()),
    },
    "variables": {
        v: {"label": lab, "units": u, "cmap": cm,
            "range": robust_range(ds[v]),
            "anomaly_range": [-max(map(abs, robust_range(anom[v]))), max(map(abs, robust_range(anom[v])))],
            "forecast": v == "mlotst"}  # flip to True for "to" once an SST forecast exists
        for v, (lab, u, cm) in VARS.items()
    },
    "forecast_months": [],  # filled in by the model notebook
}
with fs.open(f"{OUT_DIR}/meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2)[:1500])
print("Done. Outputs:", fs.ls(OUT_DIR))

{
  "first_month": "1993-01",
  "last_observed_month": "2026-08",
  "climatology_years": [
    "1993",
    "2018"
  ],
  "bounds": {
    "west": -149.9375,
    "east": -120.0625,
    "south": 40.0625,
    "north": 61.9375
  },
  "variables": {
    "to": {
      "label": "Sea surface temperature",
      "units": "\u00b0C",
      "cmap": "thermal",
      "range": [
        4.468,
        18.899
      ],
      "anomaly_range": [
        -2.081,
        2.081
      ],
      "forecast": false
    },
    "so": {
      "label": "Sea surface salinity",
      "units": "PSU",
      "cmap": "haline",
      "range": [
        31.548,
        33.168
      ],
      "anomaly_range": [
        -0.312,
        0.312
      ],
      "forecast": false
    },
    "zo": {
      "label": "Sea surface height",
      "units": "m",
      "cmap": "balance",
      "range": [
        0.217,
        0.716
      ],
      "anomaly_range": [
        -0.111,
        0.111
      ],
      "forecast": false
    },
    "ml

In [12]:
## Mini test
print(enso.loc["1997-04":"1998-06", "phase"].value_counts())   # expect mostly el_nino (1997-98 super El Niño)
print(enso.loc["2015-03":"2016-05", "phase"].value_counts())   # expect mostly el_nino (2015-16)
print(enso.loc["2010-06":"2011-04", "phase"].value_counts())   # expect mostly la_nina
print(enso["provisional"].sum(), "provisional months")          # expect 4

phase
el_nino    12
neutral     3
Name: count, dtype: int64
phase
el_nino    15
Name: count, dtype: int64
phase
la_nina    11
Name: count, dtype: int64
4 provisional months
